In [1]:
import os
import sys
import numpy as np
import pandas as pd

from datetime import datetime
from dateutil import relativedelta

import random
random.seed(123)

import plotly.graph_objs as go
import plotly.offline as pyo
import plotly.subplots as psub

# import functions
sys.path.append("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/")

from load_data import load_data
from load_spec import load_spec
from summarize import summarize

In [2]:
# pd.set_option('display.max_columns', 30)

In [3]:
## Load data
country = 'US';         # United States macroeconomic data
sample_start = datetime.strptime('2000-01-01', '%Y-%m-%d'); # estimation sample

## Load model specification and dataset.
# Load model specification structure `Spec`
Spec = load_spec('../data/0_source/Spec_US_example.xls');
# Parse `Spec`
SeriesID, SeriesName, Units, UnitsTransformed, Frequency = Spec['seriesid'], Spec['seriesname'], Spec['units'], Spec['unitstransformed'], Spec['frequency']

# Prepare data -----------------------------------------------------------
datafile = pd.read_excel("../data/02_intermediate/harmonized_time_series.xlsx", header=None)
X, Time, Z, header = load_data(datafile, Spec, sample_start);

/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/load_spec.py:41: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Table 1: Model specification
              SeriesID                   SeriesName                 Units  \
0               PAYEMS           Payroll Employment  Thousands of Persons   
1               JTSJOL                 Job Openings             Thousands   
2             CPIAUCSL         Consumer Price Index                 Index   
3              DGORDER         Durable Goods Orders           $, Millions   
4                RSAFS                 Retail Sales           $, Millions   
5               UNRATE            Unemployment Rate                     %   
6                HOUST               Housing Starts    Thousands of Units   
7               INDPRO        Industrial Production                 Index   
8              DSPIC96              Personal Income   Chained $, Billions   
9              BOPTEXP                      Exports           $, Millions   
10             BOPTIMP                      Imports           $, Millions   
11             TTLCONS        Construction Spen

In [4]:
X_df = pd.DataFrame(X, columns=header, index=Time)
Z_df = pd.DataFrame(data=Z, columns=header, index=Time)
T = Time[-100:]

fig = psub.make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                            subplot_titles=("Raw Observed Data", "Transformed data"))

## Plot raw and transformed data.
# Industrial Production (INDPRO) <fred.stlouisfed.org/series/INDPRO>
series_name = "PCEC96"
idxSeries = SeriesID.index(series_name)
# Plot raw observed data
trace1 = go.Scatter(
    x=Z_df.loc[T].sort_index()[series_name].index,
    y=Z_df.loc[T].sort_index()[series_name],
    mode="lines+markers", #if Z_df[series_name].isna().sum() else "lines",
    name='Raw Observed Data',
    line=dict(color="#000000", width=1.25),  # #7BCC62 / #68b562 / #7BB562
    marker={"size": 5, "symbol": "diamond"},
)
fig.add_trace(trace1, row=1, col=1)

# Plot transformed data
trace2 = go.Scatter(
    x=X_df.loc[T].sort_index().index,
    y=X_df.loc[T][series_name].sort_index(),
    mode='lines',
    name='Transformed Data',
    line=dict(color="#BDC1D6", width=1.25),  # #7BCC62 / #68b562 / #7BB562
)
fig.add_trace(trace2, row=2, col=1)
fig.update_layout(
    height=800,
    width=800,
    showlegend=False,
    title_text=series_name,
    plot_bgcolor="white",
)
fig.update_xaxes(range=[T[0], T[-1]], row=1, col=1, gridcolor="lightgrey")
fig.update_yaxes(
    title_text=Units[idxSeries],
    row=1,
    col=1,
    gridcolor="lightgrey"
    )

fig.update_xaxes(range=[T[0], T[-1]], title_text='Time', row=2, col=1, gridcolor="lightgrey")
fig.update_yaxes(title_text=UnitsTransformed[idxSeries], row=2, col=1, gridcolor="lightgrey")
fig.show()